# 02 — Préparation des données Alibaba 2017

Ce notebook inspecte le résultat de `ML/src/build_dataset.py`. Le script lit les instances par blocs de 100 000 lignes, puis les regroupe par `(job_id, task_id)`.

Sources : [schéma officiel](https://github.com/alibaba/clusterdata/blob/master/cluster-trace-v2017/schema.csv), [description de la trace](https://github.com/alibaba/clusterdata/blob/master/cluster-trace-v2017/trace_201708.md).

Pour régénérer les fichiers, lancer **à la racine du projet** :
```powershell
.venv\Scripts\python.exe -m ML.src.build_dataset
```
Les CSV bruts restent intacts.

In [ ]:
from pathlib import Path
import json
import pandas as pd

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "ML" / "src" / "build_dataset.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Ouvrir le notebook depuis le projet ai-cloud-optimizer.")
processed = project_root / "ML" / "data" / "processed"
report = json.loads((processed / "data_quality_report.json").read_text(encoding="utf-8"))
dataset = pd.read_csv(processed / "task_resource_dataset.csv")
dataset.shape

## Exclusions des enregistrements

Les catégories de ce tableau sont exclusives et appliquées dans l'ordre : état terminé, identifiants présents et positifs, tâche connue, horodatages de fin cohérents. Les temps de début négatifs sont conservés lorsqu'ils précèdent le début de la trace.

Une ligne source décrit une **tentative d'exécution**. Le fichier ne fournit pas d'identifiant unique d'instance ; plusieurs tentatives ne sont donc pas supprimées sous prétexte qu'elles ont la même tâche ou la même machine.

In [ ]:
pd.Series(report["instance_funnel_exclusive"], name="records").to_frame()

In [ ]:
pd.Series(report["tasks"], name="tasks").to_frame()

## Qualité CPU et RAM

Sur les tentatives retenues, les paires moyenne/maximum doivent être présentes, finies, non négatives et respecter moyenne ≤ maximum. Les diagnostics suivants peuvent se chevaucher.

Les cibles CPU et RAM sont calculées **séparément**. Une RAM manquante ne devient jamais zéro.

In [ ]:
pd.Series(report["resource_diagnostics_on_accepted_attempts_overlapping"], name="records").to_frame()

In [ ]:
dataset.head()

In [ ]:
dataset[["cpu_measurement_coverage", "mem_measurement_coverage"]].describe()

La couverture correspond à la fraction de **tentatives terminées acceptées** dont les mesures sont valides ; ce n'est pas une fraction du nombre d'instances demandées.

`mean_of_attempt_averages` est une moyenne par tentative, non pondérée par sa durée. `observed_peak` est le maximum observé parmi les tentatives mesurées. Aucun de ces indicateurs ne représente la consommation totale simultanée d'une tâche.

In [ ]:
features = report["model_contract"]["features"]
cpu_targets = report["model_contract"]["targets"]["cpu"]
mem_targets = report["model_contract"]["targets"]["mem"]
cpu_data = dataset.loc[dataset["eligible_cpu"], ["job_id", "task_id"] + features + cpu_targets]
mem_data = dataset.loc[dataset["eligible_mem"], ["job_id", "task_id"] + features + mem_targets]
pd.DataFrame({"CPU": [len(cpu_data)], "RAM": [len(mem_data)]}, index=["Tâches utilisables"])

In [ ]:
cpu_data[features + cpu_targets].describe()

In [ ]:
mem_data[features + mem_targets].describe()

In [ ]:
assert not dataset.duplicated(["job_id", "task_id"]).any()
assert len(dataset) == report["tasks"]["output"]
assert len(cpu_data) == report["tasks"]["eligible_cpu"]
assert len(mem_data) == report["tasks"]["eligible_mem"]
assert not cpu_data[features + cpu_targets].isna().any().any()
assert not mem_data[features + mem_targets].isna().any().any()
print("Contrôles de cohérence réussis.")

## Prochaine étape : premier modèle de référence

- Entrées autorisées : `instance_num`, `plan_cpu`, `plan_mem`.
- Cibles : les consommations observées indiquées ci-dessus, avec un modèle CPU et un modèle RAM séparés.
- Séparer apprentissage/test **par `job_id`** pour éviter que des tâches du même job se retrouvent des deux côtés.
- Les identifiants, dates, états, mesures réelles et couvertures ne seront pas des entrées du modèle.
- Comparer d'abord une prédiction constante et un modèle simple, avec une évaluation selon la couverture des mesures.

Les valeurs CPU restent celles des fichiers ; la RAM reste normalisée. La conversion vers des tailles de VM et le lien avec `expected_users` dans l'API restent à définir. Les tâches terminées et disposant de mesures constituent un sous-ensemble potentiellement biaisé de la trace.